In [ ]:
import os
import pandas as pd 
import numpy as np
import xarray as xr
import string

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import ttest_ind
from scipy import stats
from scipy.stats import pearsonr
from collections import defaultdict
import matplotlib.ticker as mticker

from tc_basins import BASINS
from gpi_reader import GPIReader  # You must define this elsewhere
from TrackUtil import IBTrACSAnalyzer
from sklearn.utils import resample
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
import matplotlib.patches as mpatches  # At top if not already imported


In [ ]:
class GPIMetricEvaluator:
    def __init__(self, data_dir, exp_dict, syear, eyear, varname="GPI", obs_label=None, ref_label="ERA5"):
        self.data_dir = data_dir
        self.exp_dict = exp_dict
        self.syear = syear
        self.eyear = eyear
        self.varname = varname
        self.obs_label = obs_label
        self.ref_label = ref_label
        self.gpi_data = {}
        self.genesis_dict = None
        self.color_map = {
            "ERA5": "black",
            "NDGUVTQ_SRF1": "red"
        }
        self.monthly_clim = {}
        self.annual_means = {}

    def load_all_data(self):
        for exp in self.exp_dict:
            reader = GPIReader(self.data_dir, exp, self.syear, self.eyear, gpi_vars=[self.varname])
            reader.load_data()
            da = reader.get_data(self.varname)
            if exp != 'ERA5':
                da['time'] = da.indexes['time'] - pd.to_timedelta(15, unit='D')
            self.gpi_data[exp] = da

    def set_genesis_data(self, genesis_dict):
        self.genesis_dict = genesis_dict
        
    def normalize_time_to_month_start(self, da):
        """Normalize time to the first day of each month."""
        da = da.copy()
        da['time'] = xr.DataArray(
            pd.to_datetime({
                "year": da.time.dt.year.values,
                "month": da.time.dt.month.values,
                "day": 1
            }),
            dims="time"
        )
        return da
    
    def compute_annual_mean(self):
        self.annual_means = {}

        # Normalize time
        normalized_data = {
            exp: self.normalize_time_to_month_start(da)
            for exp, da in self.gpi_data.items()
        }

        # Find common time range across experiments
        common_start = max([da.time.min().values for da in normalized_data.values()])
        common_end = min([da.time.max().values for da in normalized_data.values()])
        common_start = pd.to_datetime(common_start).replace(month=1, day=1)
        common_end = pd.to_datetime(common_end).replace(month=12, day=31)

        for exp, da in normalized_data.items():
            da_clipped = da.sel(time=slice(common_start, common_end))
            if len(da_clipped.time) == 0:
                print(f"[WARNING] No valid time range for {exp} after clipping.")
                self.annual_means[exp] = xr.full_like(da.isel(time=0), np.nan)
            else:
                self.annual_means[exp] = da_clipped.resample(time="1Y").mean("time")
            
    def compute_monthly_climatology(self):
        self.monthly_clim = {
            exp: da.groupby("time.month").mean("time") for exp, da in self.gpi_data.items()
        }

    def regional_average(self, da, region):
        lon1, lon2 = region["lon1"], region["lon2"]
        lat1, lat2 = region["lat1"], region["lat2"]
        if lon1 < lon2:
            subset = da.sel(lat=slice(lat1, lat2), lon=slice(lon1, lon2))
        else:
            subset1 = da.sel(lat=slice(lat1, lat2), lon=slice(lon1, 180))
            subset2 = da.sel(lat=slice(lat1, lat2), lon=slice(-180, lon2))
            subset = xr.concat([subset1, subset2], dim="lon")
        return subset.mean(dim=("lat", "lon"))

    def plot_all_basin_seasonal_cycles(self, region_dict, figsize=(16, 10), ncols=3, fontz=10, save_path=None, use_month_abbrev=True):
        """Plot seasonal cycles of GPI and genesis for all basins using twin y-axes."""
        self.compute_monthly_climatology()
        n_basins = len(region_dict)
        nrows = int(np.ceil(n_basins / ncols))
        fig, axs = plt.subplots(nrows, ncols, figsize=figsize)
        axs = axs.flatten()

        # Assign distinct colors to experiments
        default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        used_keys = set(self.color_map.keys())
        available_exps = [exp for exp in self.exp_dict if exp not in used_keys]
        available_colors = [c for c in default_colors if c not in ["black", "red"]]
        for exp, color in zip(available_exps, available_colors):
            self.color_map[exp] = color

        panel_labels = list(string.ascii_lowercase)
        month_ticks = list(range(1, 13))
        month_labels = ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'] if use_month_abbrev else list(map(str, month_ticks))

        bar_handle = None

        for i, (basin, region) in enumerate(region_dict.items()):
            ax1 = axs[i]
            ax2 = ax1.twinx()

            for exp, clim in self.monthly_clim.items():
                color = self.color_map.get(exp, None)
                reg_mean = self.regional_average(clim, region)
                ax1.plot(month_ticks, reg_mean, label=self.exp_dict[exp], linewidth=1.8, color=color)
                ax1.yaxis.set_major_locator(MaxNLocator(nbins=6, prune='both'))
                ax1.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

            # Observed genesis bars
            if self.genesis_dict and self.obs_label in self.genesis_dict:
                obs = self.genesis_dict[self.obs_label]
                if "basin" in obs.dims and basin in obs.basin.values:
                    obs_basin = obs.sel(basin=basin)
                    obs_series = obs_basin.groupby("time.month").sum("time")
                    bar_container = ax2.bar(month_ticks, obs_series, alpha=0.3, color="gray")
                    bar_handle = mpatches.Patch(color="gray", alpha=0.3, label="Observed Genesis")
                    ax2.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True, prune='both'))

                    # === Compute and add correlation (ERA5 GPI vs Observed Genesis) ===
                    obs_clim = obs_basin.groupby("time.month").mean("time")
                    if self.ref_label in self.monthly_clim:
                        gpi_clim = self.regional_average(self.monthly_clim[self.ref_label], region)
                        try:
                            corr_val, _ = pearsonr(gpi_clim.values, obs_clim.values)
                            corr_str = f"r = {corr_val:.2f}"
                        except Exception:
                            corr_str = "r = NaN"
                        if basin in ['Southwest Indian Ocean','Southeast Indian Ocean', 'South Pacific', 'Southern Hemisphere']:
                            xloc = 0.6
                            yloc = 0.95
                        else:
                            xloc = 0.05
                            yloc = 0.95
                        ax1.text(
                            xloc, yloc, corr_str,
                            transform=ax1.transAxes,
                            fontsize=fontz,
                            verticalalignment='top',
                            bbox=dict(facecolor='white', edgecolor='gray', alpha=0.6, boxstyle='round,pad=0.2')
                        )

            # Title and labels
            panel_tag = f"({panel_labels[i]}) {basin}"
            ax1.set_title(panel_tag, fontsize=fontz)
            ax1.set_xlabel("Month", fontsize=fontz)
            ax1.set_ylabel("GPI", fontsize=fontz)
            ax2.set_ylabel("Total Count", fontsize=fontz)
            ax1.set_xticks(month_ticks)
            ax1.set_xticklabels(month_labels)
            ax1.tick_params(labelsize=fontz)
            ax2.tick_params(labelsize=fontz)
            ax1.set_ylim(bottom=0)
            ax2.set_ylim(bottom=0)
            ax1.grid(True, linestyle='--', alpha=0.5)

        # Remove unused subplots
        for j in range(i + 1, len(axs)):
            fig.delaxes(axs[j])

        #fig.suptitle("Seasonal Cycle of GPI and TC Genesis (All Basins)", fontsize=fontz + 4)
        plt.tight_layout(rect=[0, 0, 1, 0.94])

        # === Shared Legend ===
        unique = {}
        for exp, label in self.exp_dict.items():
            color = self.color_map.get(exp, "gray")
            unique[label] = plt.Line2D([0], [0], color=color, lw=1.8)
        if bar_handle:
            unique["Observed Genesis"] = bar_handle

        fig.legend(
            unique.values(), unique.keys(),
            loc='lower center', bbox_to_anchor=(0.5, -0.08),
            ncol=8, fontsize=fontz, frameon=True, edgecolor='gray'
        )

        if save_path:
            fig.savefig(save_path, bbox_inches="tight", dpi=300)
            print(f"[INFO] Saved seasonal cycle figure to: {save_path}")

        plt.show()

    def plot_gpi_genesis_interannual_and_climatology_corr(
            self, 
            region_dict, 
            ref_label="ERA5",
            season_months=None, 
            n_bootstrap=None, 
            confidence=None
        ):
        self.compute_monthly_climatology()
        self.compute_annual_mean()

        basin_labels = list(region_dict.keys())
        exps = list(self.gpi_data.keys())
        interannual_corrs = {exp: [] for exp in exps}
        climatology_corrs = {exp: [] for exp in exps}

        for basin in basin_labels:
            region = region_dict[basin]

            if self.genesis_dict and self.ref_label in self.genesis_dict:
                obs = self.genesis_dict[self.ref_label]
                if "basin" in obs.dims and basin in obs.basin.values:
                    obs_basin = obs.sel(basin=basin)
                    obs_annual = obs_basin.resample(time="1YE").sum("time")
                    obs_clim = obs_basin.groupby("time.month").mean("time")

                    for exp in exps:
                        gpi_annual = self.regional_average(self.annual_means[exp], region)
                        gpi_clim = self.regional_average(self.monthly_clim[exp], region)

                        try:
                            corr_inter, _ = pearsonr(gpi_annual.values, obs_annual.values)
                            corr_clim, _ = pearsonr(gpi_clim.values, obs_clim.values)
                        except Exception:
                            corr_inter, corr_clim = np.nan, np.nan

                        interannual_corrs[exp].append(corr_inter)
                        climatology_corrs[exp].append(corr_clim)
                else:
                    for exp in exps:
                        interannual_corrs[exp].append(np.nan)
                        climatology_corrs[exp].append(np.nan)
            else:
                for exp in exps:
                    interannual_corrs[exp].append(np.nan)
                    climatology_corrs[exp].append(np.nan)

        # === Plot ===
        fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

        x = np.arange(len(basin_labels))
        bar_width = 0.8 / len(exps)

        for i, exp in enumerate(exps):
            color = self.color_map.get(exp, None)
            axs[0].bar(x + i * bar_width, interannual_corrs[exp], width=bar_width, label=self.exp_dict.get(exp, exp), color=color)
            axs[1].bar(x + i * bar_width, climatology_corrs[exp], width=bar_width, label=self.exp_dict.get(exp, exp), color=color)

        axs[0].set_title("Interannual Correlation (Annual Mean)", fontsize=12)
        axs[1].set_title("Climatology Correlation (Monthly Cycle)", fontsize=12)

        for ax in axs:
            ax.set_xticks(x + bar_width * (len(exps) - 1) / 2)
            ax.set_xticklabels(basin_labels, rotation=45)
            ax.set_ylabel("Correlation Coefficient")
            ax.set_ylim(-0.5, 1)
            ax.grid(True, linestyle="--", alpha=0.4)

        #fig.suptitle("GPI–Genesis Correlation by Basin", fontsize=14)
        handles = []
        labels = []

        for exp, color in self.color_map.items():
            if exp in self.exp_dict and self.exp_dict[exp] not in labels:
                handles.append(plt.Line2D([0], [0], color=color, lw=8))
                labels.append(self.exp_dict[exp])

        fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.05), ncol=4, frameon=True)
        
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()


In [ ]:
if __name__ == "__main__":
    # === Paths ===
    top_path = "/anvil/scratch/x-szhang3"
    data_path = f"{top_path}/post_data/TempestExtremes/TCS/GPI/output"
    ibtracs_path = f"{top_path}/post_data/TempestExtremes/TCS/OBS/tc_track/IBTrACS/IBTrACS.ALL.v04r00.nc"
    out_path = "/home/x-szhang3/nudging_analysis/nudging_evaluation/data/TempestExtremes/IBTrACS" 
    fig_path = "./"    
    os.makedirs(out_path, exist_ok=True)

    # === Time Range ===
    syear, eyear = 2008, 2017
    basin_nc_path = os.path.join(out_path, f"IBTrACS_basin_monthly_timeseries_{syear}-{eyear}.nc")
    #os.remove(basin_nc_path)
    
    # === Experiments ===
    exp_dict = {
        "ERA5": "ERA5",
        "CLIM": "CLIM",
        "NDGUV": "NDG-UV",
        "NDGUVT": "NDG-UVT",
        #"NDGUVTQ": "NDG-UVTQ",
        "NDGUVTQ_SRF1": "NDG-UVTQ_SRF1"
    }
    
    # === Step 1: Load or compute IBTrACS basin time series ===
    if os.path.exists(basin_nc_path):
        print(f"[INFO] Using existing IBTrACS basin timeseries file: {basin_nc_path}")
        ds_basin = xr.open_dataset(basin_nc_path)
    else:
        print("[INFO] Basin time series file not found — computing from IBTrACS...")
        analyzer = IBTrACSAnalyzer(ibtracs_path=ibtracs_path, syear=syear, eyear=eyear)
        analyzer.load_and_filter_ibtracs()
        genesis_df, density_df = analyzer.get_basin_monthly_genesis_and_density_timeseries()
        analyzer.save_basin_timeseries_to_netcdf(genesis_df, density_df, out_path=basin_nc_path)
        ds_basin = xr.open_dataset(basin_nc_path)

    # === Convert IBTrACS genesis to xarray DataArray ===
    genesis_da = xr.DataArray(
        data=ds_basin["genesis_count"].values,
        coords={"time": ds_basin["time"].values, "basin": ds_basin["basin"].values},
        dims=["time", "basin"],
        name="genesis_count"
    )

    # === Step 2: Load GPI data and run diagnostics ===
    evaluator = GPIMetricEvaluator(
        data_dir=data_path,
        exp_dict=exp_dict,
        syear=syear,
        eyear=eyear,
        varname="GPI",
        obs_label="IBTrACS",
        ref_label="ERA5"
    )

    evaluator.load_all_data()
    evaluator.set_genesis_data({"IBTrACS": genesis_da})
    evaluator.compute_annual_mean()
    evaluator.compute_monthly_climatology()

    # === Plot seasonal cycle (GPI and Genesis with twin y-axis) ===
    fig_name = "Figure_09.pdf"
    if fig_name is not None:
        output_file = os.path.join(fig_path,fig_name)
    else:
        output_file = os.path.join(fig_path,"Figure_gpi_seasonal_cycle_all_basins.pdf")
    
    evaluator.plot_all_basin_seasonal_cycles(
        BASINS,
        figsize=(18, 11),
        save_path = fig_name, 
        fontz=18
    )

#    # === Plot interannual and annual-cycle correlation between GPI and Genesis ===
#    evaluator.plot_gpi_genesis_interannual_and_climatology_corr(
#        ref_label="ERA5",
#        region_dict=BASINS,
#        season_months=[1,2,3,4,5,6, 7, 8, 9, 10,11,12],
#        n_bootstrap=1000,
#        confidence=0.95
#    )
